# Implementing a simple Retrieval-augmented generation (RAG)
This Lab is based on a workshop created by our friends of the [Swiss-AI-Center](https://www.hes-so.ch/swiss-ai-center).

Original Authors:
- Jonathan Guerne, Research assistant at Haute Ecole Arc Ingénierie, Switzerland
- Henrique Marques Reis, Research assistant at Haute Ecole Arc Ingénierie, Switzerland
- Célien Donzé, Scientific Collaborator at HEIA-FR, Switzerland

Adapted by:
- Francesco Carrino, Professor at Haute Ecole Ingénierie, Valais/Wallis Switzerland

## Introduction
This notebook explains how to create a RAG (Retrieval Augmented Generation) system to answer questions about online or PDF documents. We will use a self-hosted LLM with Ollama to generate answers to the questions. Additionally, we will use a vector database to retrieve relevant documents for answering the questions.

### Documentation and references
Technologies and tools used in this lab:
- [Large Language Model (LLM)](https://en.wikipedia.org/wiki/Large_language_model), a model that generates text (typically) baseded on the transformer architecture. In this lab, we will used opensource models like LLAMA 3.x.
- [Ollama](https://ollama.com/): Ollama is a platform that democratizes access to LLMs by enabling users to run them locally on their machines. In this lab, it will be used to run the LLMs on your machine or in the cloud (Kaggle).
- [Vector database](https://en.wikipedia.org/wiki/Vector_database): A Vector Database is a relational database system specifically designed to process vectorized data. Unlike conventional databases that contain information in tables, rows, and columns, vector databases work with vectors–arrays of numerical values that signify points in multidimensional space. When used with text data, vector databases can store and retrieve information based on the semantic meaning of the text, rather than just the text itself. Through a process called embedding, text data is converted into vectors that represent the **meaning** of the text. This meanse text with similar meaning is stored close to each other in the vector space. We will use [FAISS](https://python.langchain.com/docs/integrations/vectorstores/faiss/) an opensource library for efficient similarity search and clustering of dense vectors. In this lab, it will be used to store information about pdf and web documents.
- [LangChain](https://python.langchain.com/docs/introduction/): LangChain is a framework for developing applications powered by large language models (LLMs).  It is based on the idea of "chains", where a chain defines sequences of actions, where each step can involve querying an LLM, prompt management, manipulating data, or interacting with external tools applications. In this lab, it will be used to put together the LLM and the vector database to create a RAG system.
- [Retrieval-augmented generation (RAG)](https://en.wikipedia.org/wiki/Retrieval-augmented_generation): RAG is a technique to improve LLMs by adding a retrieval component from an external data source. This is the main goal of this tutorial.

### Structure and goals
This lab is divided into 4 parts. The first 3 parts, you will be guided to install and implement a simple RAG working on pdf files. Your goal in this part understand the code. We provide some questions to guide your learning through the code.

The last part is a challenge where you will have to use what you learnt to implement a RAG working on web pages.

1. **Installation**: Install the necessary libraries and download the data. We provide two guides: one for running the code on Kaggle (or Google Colab), and another for running the code locally.
2. **Creating and manipulating prompts**: We will create prompts and create simple query the LLM and the vector database.
3. **Creating a RAG system**: We will create a simple RAG system to answer questions about pdf documents. The pdf will be embedded in the vector database and the LLM will generate the answers.
4. **Challenge**: In this last part, *you* will implement a RAG system to answer questions about web pages.

---

## 1. Installation



If your machine is not very performant, you can use Kaggle or Google Colab to run this notebook. The following code block installs the necessary packages.
If you choose to run the code on Kaggle, login to your account and import this notebook. It is raccomanded to have a verified account to have access to the GPU.

Below you can find the instraction to install the necessary packages either on Kaggle or on your local machine.

### Installing Ollama (Kaggle or Google Colab)

If you are in Kaggle or Google Colab, you can install Ollama using the following code block. If want to run this notebook locally, see the next section.

These instructions are based on the Medium post titled [How to Run Ollama Models on Google Colab and Kaggle: A Complete Guide](https://medium.com/@mradul18varshney/how-to-run-ollama-generative-ai-models-on-google-colab-and-kaggle-a-complete-guide-3e01fc12512f).

In [ ]:
# Package installation (Kaggle or Google Colab)
!pip install langchain langchain_huggingface langchain-community faiss-cpu pymupdf pypdf sentence_transformers rich wget python-dotenv cryptography unstructured





NOTE: you may receive an error message such as `ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.`. This is a known issue and can be ignored.

#### Setting up the environment (Kaggle or Google Colab)

This update package list and install curl, which then use to fetch Ollama installation script.

In [ ]:
# Update system files and install curl
!apt-get update && apt-get install -y curl

#### Installing Ollama (Kaggle or Google Colab)

This will install Ollama automatically.

In [ ]:
# Install Ollama
!curl https://ollama.ai/install.sh | sh

#### Starting the Ollama server (Kaggle or Google Colab)

Here, we use Python’s `subprocess.Popen()` to run the Ollama server in the background, listening on the specified host and port (127.0.0.1:11438). The server will be available for communication via the API.

You will need this address later for the RAG model.

In [ ]:
import subprocess
import time

# Set the environment variable for the Ollama host and port
os.environ['OLLAMA_HOST'] = '127.0.0.1:11438'

# Start the Ollama server in a subprocess
ollama_process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Wait for the server to start
print("Starting Ollama server...")
time.sleep(1)  # Wait for the server to initialize

print("Ollama server is running on 127.0.0.1:11438")

#### Pulling an open-source LLM model (Kaggle or Google Colab)
This command downloads the llama3.2:1b model from Ollama’s servers (a small version of LLAMA 3.2), making it available for use on your notebook machine environment.
Also, refer here https://ollama.com/search for checking the list of models available on Ollama model repository.

In [ ]:
# Pull a model (e.g., llama)
print("Pulling the llama model...")
subprocess.run(['ollama', 'pull', 'llama3.2:1b'])

#### Interacting with the model (Kaggle or Google Colab)

To test the installation and the model, we can use the following code block to interact with the model.

In [ ]:
import requests
import json

def generate_response(query):
    # Define the API URL for the same host as mentioned when starting the server.
    url = "http://localhost:11438/api/chat"
    
    # Define the payload (data) to send in the POST request
    payload = {
        "model": "llama3.2:1b",
        "messages": [
            {
                "role": "user",
                "content": query
            }
        ],
        "keep_alive": 0,
        "stream": False
    }
    
    headers = {
        "Content-Type": "application/json"
    }
    
    # Make the POST request to the API
    try:
        response = requests.post(url, data=json.dumps(payload), headers=headers)
    
        # Check if the request was successful (status code 200)
        if response.status_code == 200:
            output = response.text
            return output
        else:
            print(f"Failed to get a response. Status code: {response.status_code}")
            return ("Error response:", response.text)
    except Exception as e:
        print(f"An error occurred: {e}")

Once the server is running and the model is pulled, you can start interacting with the model using Python code. In this example, the generate_response() function sends a POST request to the Ollama API, passing in the user query. The model processes the query and returns a generated response.

*Note: If you change the model you will get the error that the model is not downloaded and you have to pull it using ollama pull. Also, check the model name to match with the model you pulled using `ollama pull` command.*

In [ ]:
# Using the function to get the output
# The output is in JSON format, and you can parse it using Python’s json module.

query = "What is the difference between AI and ML? Give a short answer."
output = generate_response(query)
json.loads(output)['message']['content']

#### Using Ollama’s Python SDK (Kaggle or Google Colab)

To interact with the Ollama server, you can use the Python SDK. The SDK is available on PyPI and can be installed using pip.

In [ ]:
!pip install ollama

The code below interacts with the Ollama model using the Python SDK to send a query and print the generated response.

*Note: (again) If you change the model you will get the error that the model is not downloaded and you have to pull it using ollama pull.*

In [ ]:
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(model='llama3.2:1b', messages=[
  {
    'role': 'user',
    'content': 'Show me the lyrics of any song?',
  },
])
print(response['message']['content'])

If all works, you should see the generated response from the model. In my test, I got the lyrics from the song "Yesterday" by The Beatles. What did you get?

- - - 

### Installing Ollama (locally on your machine)

This guide will help you to install Ollama on your local machine. You need a fairly good machine to run LLMs models and space in your hard disk to store the LLM models (several GB).  If you have a machine with a GPU, you can run the models faster.
However, there are lighter models that can run on CPU and provide results good enough for this lab and many applications (e.g., you can try "llama3.2:1b", or "qwen2.5:0.5b" model).

#### Download and install Ollama
Ollma is available for macOS, Linux, and Windows. Go to the official [website](https://ollama.com/) and download the version for your operating system. Follow the instructions to install it.

#### Running Ollama
After installing Ollama, you should see the Ollama icon in your system tray. You can use it to see the logs and quit the application.

Usuall, after the installation Ollama is already running. 

If you want to run it again, you can run the following command in your terminal:

```bash
ollama serve
```

To run Ollama, go to the [Ollama model list](https://ollama.com/search) and select a model suitable for your machine. Start humble with a small model. You can for instance try the small version of [LLAMA 3.2](https://ollama.com/library/llama3.2).

Follow the instractions for the model you selected. For example, for LLAMA 3.2 (the 1B parameters version), run in your terminal:

```
ollama run llama3.2:1b
```

The `run` command will pull (download) the model to your machine and then run it, exposing it via the API started with `ollama serve`.

If you want to *pull* a differen model, you can run:

```bash
ollama pull <model_name>
```

The commanda `ollama list` will show you the list of models available on your machine.

To switch between models available on your machine, you can just use the `run` command (NOTE: if the model is not present in your machine, the `run` command will autamtically try to pull it). The `stop` command will stop a running model.

#### Get Ollama address
In order to interact with the model, you need to know the address where the model is running. Ollama binds to the local address 127.0.0.1 on port 11434 by default.

This means that you can note http://localhost:11434

### Package installation (with Poetry)
We will install the necessary packages using Poetry. We provide a `pyproject.toml` file with the necessary dependencies. We assume you have [Poetry installed](https://github.com/hei-synd-aml/lab-0-TutoPoetry) on your machine. Open a terminal in the project location and run the following command to install the packages.

```bash
poetry install
```

---

## 2. Creating a prompt and interacting with the model

OK, now that you have the Ollama server running and the model pulled, let's start by creating a prompt and interacting with the model.
In this section, we will create a prompt and interact with the model to generate text.

### Imports

In [1]:
import os
from pathlib import Path

import wget
from dotenv import load_dotenv
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain_ollama import OllamaLLM
from rich.console import Console
from rich.markdown import Markdown

console = Console()

### Constants

In [2]:
load_dotenv(override=True)

OLLAMA_ADDRESS = "http://localhost:11434"  # replace with your OLLAMA address, usually http://localhost:11434
LLM_NAME = "llama3.2:1b"  # replace with the LLM name you pulled from the OLLAMA


### Connecting to LLM

NOTE: if you get "ConnectionError", check if the Ollama server is running and the address is correct.

In [3]:
llm = OllamaLLM(
    model=LLM_NAME,
    base_url=OLLAMA_ADDRESS,
    temperature=0.1,  # Will be explained later
    stop=["<end_of_turn>"],
)

llm.generate(["Hello, how are you today?"]).generations[0][0].text

"I'm doing well, thanks for asking. I'm a large language model, so I don't have feelings or emotions like humans do, but I'm here and ready to help you with any questions or topics you'd like to discuss. How about you? How's your day going so far?"

*** Question**:
- What is the role of the temperature parameter in a LLM?

**Answer**:
The temperature parameter in a language model (LLM) controls the randomness of the model's output.

A lower temperature value (closer to 0) makes the model more deterministic, favoring higher probability words and resulting in more predictable and repetitive text.

A higher temperature value (closer to 1) increases randomness, allowing for more creative and diverse responses by giving less probable words a better chance of being chosen.

Adjusting the temperature helps balance between coherence and creativity in the generated text.

### Creating a prompt

A prompt is generally divided into two parts: the context and the question.

The context provides the information that the model will use to generate its answer, while the question specifies what the model is expected to respond to.
Typically, the context contains a system prompt, which explains the expected behavior from the model, and other additional information that can help the model to provide the right answer. Typically, when used for chatbots, in the context there is a summary (or the entierity!) of all the previous exchange the user had with the model in that conversation. 

LangChain uses templates and markers indicating where to insert the user's question and the context within the template. For more infomation, you can check [Langchain prompt templates documentation](https://python.langchain.com/v0.2/docs/concepts/#prompt-templates).

In [4]:
# A langchain's template is nothing more than a string with {markers} indicating placeholder where other text will be inserted when creating a prompt.

template = """
You are an helpful assistant that answer the question in detail.

Human input: {question}
Assistant:"""

prompt = PromptTemplate(input_variables=["question"], template=template)
prompt

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='\nYou are an helpful assistant that answer the question in detail.\n\nHuman input: {question}\nAssistant:')

### Creating the chain and start a conversation

In [5]:
# Simple chain putting together a prompt and a model 
simple_llm_chain = prompt | llm

In [6]:
result = simple_llm_chain.invoke(input="When is held the conference called AI-days 2025?")

console.print(Markdown(result)) # we print the output proposed by the LLM

I can provide you with information on when the AI Days 2025 conference might be held. However, please note that I'm
a large language model, I don't have real-time access to specific event calendars or schedules.                    

That being said, I can suggest some possible ways for you to find out when the AI Days 2025 conference is scheduled
to take place:                                                                                                     

 1 Check the official website: You can visit the official website of the organizers of the conference (e.g., [AI   
   Days 2025 website]) and look for a "Conference Schedule" or "Event Calendar" section.                           
 2 Search online event calendars: Websites like Eventbrite, Meetup, or other online event calendars might list     
   upcoming conferences related to AI and machine learning.                                                        
 3 Contact the organizers directly: You can try reaching out to the organizers of the conference via email or phone
   to inquire about the schedule for AI Days 2025.                                                                 

Unfortunately, I couldn't find any information on an "AI Days" conference that is widely recognized or scheduled   
for 2025. If you have more context or details about the conference, I'd be happy to try and help further!

**Question**:
- What is the relationship between the `input` parameter in `llm_chain.invoke(input="")` and the prompte template defined above?

**Answer**:
- The `input` parameter is used to pass the question to the model. The question is inserted using the `{question}` marker.
NOTE: different LLMs may have different markers for the question and context. Check the documentation of the model you are using. This is a true challenge of prompt-engineering.

NOTE: We asked a quite specific question. It is not surprising that the model did not provide a good answer. The model has not access to the internet and this information is problably not part of its trainig.

What we can do to improve the answer? **RAG**, of course!

---

## 3. Using the RAG

For the moment, we have a model that can generate answers to questions. To provide answers, it only uses its knolwege got at training time and the context we provide. But what if we could provide the model with a set of documents and ask it to retrieve the most relevant ones to answer the question?
That is the idea behind the Retrieval-Augmented Generation (i.e., literally, we augment the generation of text by enriching the context with information contained in relevant documents).

<img src="./img/RAG_schema.png" alt="Generic schema of a RAG system" width="600"/>

[*(Source of the image)*](https://www.promptingguide.ai/research/rag)

### Downloading the pdfs

In [7]:
# Create the "data/PDFs" folder if it doesn't exist
PDF_FOLDER = Path("data/PDFs")
os.makedirs(PDF_FOLDER, exist_ok=True)

urls = [
    "https://ai-days.swiss-ai-center.ch/PDF/Session1.pdf",
    "https://ai-days.swiss-ai-center.ch/PDF/Session2a.pdf",
    "https://ai-days.swiss-ai-center.ch/PDF/Session2b.pdf",
    "https://ai-days.swiss-ai-center.ch/PDF/Session3a.pdf",
    "https://ai-days.swiss-ai-center.ch/PDF/Session3b.pdf",
]

# Download the PDFs
for url in urls:
    name = url.split("/")[-1]
    if not (PDF_FOLDER / name).is_file():
        filename = wget.download(url, f"data/PDFs/{name}")
console.print("Pdf file downloaded successfully.", style="bold green")

Pdf file downloaded successfully.

### Loading and preprocessing PDF files

In [8]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '(?<=\. )', '(?<=\, )', ' ', '']
)

***Questions***
- Investigate and explain the role of `chunk_size` and `chunk_overlap` in the code above.
- What are the advantages and disadvantages of having bigger/smaller `chunk_size`?
- What are the advantages and disadvantages of having bigger/smaller `chunk_overlap`?
- What is a difference between a token and a chunk?
- What is the parameter `separators` used for?
- What does the **Recursive** Character Text Splitter does differently compared to a **Simple** character text splitter? 

***Answers***
- The `chunk_size` parameter defines the size of the chunks in which the text is divided. A larger chunk size will result in fewer chunks, while a smaller chunk size will result in more chunks. The `chunk_overlap` parameter defines the overlap between chunks. A larger overlap will result in more redundancy between chunks, while a smaller overlap will result in less redundancy.
- Advantages of having a bigger `chunk_size` include faster processing and reduced memory requirements, while disadvantages include reduced granularity and less accurate results. Advantages of having a smaller `chunk_size` include increased granularity and more accurate results, while disadvantages include slower processing and increased memory requirements.
- Advantages of having a bigger `chunk_overlap` include increased redundancy and improved performance, while disadvantages include increased memory requirements and slower processing. Advantages of having a smaller `chunk_overlap` include reduced redundancy and improved accuracy, while disadvantages include reduced performance and increased memory requirements.
- A token is the smallest unit of text that the model can process. A chunk is a larger unit of text that is used to divide the text into manageable pieces. Chunks are used to reduce the memory requirements of the model and improve performance.
- The `separators` parameter is used to define the characters that separate chunks. By default, the separator is set to `["\n\n", "\n", " ", ""]`, which means that chunks are separated by two newline characters. This allows the model to identify the boundaries between chunks and process them separately.$
- The **Recursive** Character Text Splitter tries to split the text recursively into smaller chunks based on separators and until the chunk size is reached. This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text ([source](https://python.langchain.com/docs/how_to/recursive_text_splitter/)).

In [9]:
# All data will be in the list documents
documents=[]

In [10]:
from langchain.document_loaders import DirectoryLoader, PyPDFLoader
# Load and process the text files
# loader = TextLoader('single_text_file.txt')
loader = DirectoryLoader(PDF_FOLDER, glob="./*.pdf", loader_cls=PyPDFLoader)

pdf_docs = loader.load()
len(pdf_docs)

# tokenize pdf
documents.extend(text_splitter.split_documents(pdf_docs))
len(documents)

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 23 0 (offset 0)
Ignoring wrong pointing object 25 0 (offset 0)
Ignoring wrong po

44

#### What is a Document?
In LangChain, a  [document](https://python.langchain.com/api_reference/core/documents/langchain_core.documents.base.Document.html) is composed of two elements: the text (contained in the field `page_content`) and the metadata (contained in the field `metadata`).

The metadata is a dictionary that can contain any information you want to store about the document. In this case, we store the URL of the website.

In [11]:
# To understand what is going on, we print a document. The text:
print(documents[1].page_content)

Session 1: Compute Infrastructures for IA applications in the wild Location: Movie theater 6 With the advent of Chatbots, LLMs and other generative IA technologies, as well as other progresses in the IA ﬁeld, there is an explosion of the demand for compute force. IA is no longer computer science: it is computational science. As such, it can no longer be done with casual, self-managed equipment. More advanced compute infrastructures are required both to satisfy user needs (in terms of compute


In [12]:
# To understand what is going on, we print a document. The text:
print(documents[1].metadata)

{'producer': 'macOS Version 15.1.1 (Build 24B91) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20250116093703Z00'00'", 'title': 'Workshop_descriptions', 'moddate': "D:20250116093703Z00'00'", 'source': 'data\\PDFs\\Session1.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


**Questions**:
- why is important to also store metadata when doing RAG? Any idea?

**Answer**:
- Metadata are very important. They usually contains information useful to link a chunk of text, to its source. Therefore, metadata allow being sure that a model is not hallucinating and navigate back to the source of the information.

### Embedding a PDF in a vectorstore

In [13]:
VECTORSTORES_DIR = Path("data/vectorstores_pdf")

In [14]:
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5" # locally you can also use smaller models. Check the HuggingFace model hub for more models with different sizes, performance and languages: https://huggingface.co/spaces/mteb/leaderboard

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    # to use the "cuda" configuration, you need an nvidia GPU, and to install 
    # On Kaggle, you set to use as accelrator GPU P100 (you need a verified account)
    # model_kwargs={"device": "cpu"}, # change to cuda -> cpu if you do not have an nvdia GPU
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

c:\Users\ingno\AppData\Local\pypoetry\Cache\virtualenvs\lab-04-rag-K-_h4Q_l-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The instructions below embeds the documents in the vectorstore. Then, we can save the vectorstore to disk. In this way, we can load the vectorstore and the documents at a later time without having to re-embed the documents. If we get new relevant documents, we can add them to the vectorstore and re-embed them. If interested, you can check the [FAISS documentation](https://python.langchain.com/api_reference/community/vectorstores/langchain_community.vectorstores.faiss.FAISS.html) for more information.

In [15]:
vectorstore = FAISS.from_documents(documents=documents, embedding=embedding_model)

In [16]:
vectorstore.save_local(VECTORSTORES_DIR)

### Loading a vectorstore
We already have the vectorstore with the pdfs embedded in the variabale `vectorstore`. For completeness, in the code above, we show how to load the vectorstore from disk.

In [17]:
vectorstore = FAISS.load_local(
    VECTORSTORES_DIR, embedding_model, allow_dangerous_deserialization=True
)

### New prompt

In RAG we need to add another marker to indicate where the new information (or context) should be inserted for this we use the variable named `{context}`.

In [18]:
prompt = """
Use the following pieces of context to answer the question at the end.
Don't try to make up an answer and only use the information you know.
Use three sentences maximum and keep the answer as concise as possible.
You must answer in english. If you don't know the answer, say "I don't know".
Context:
{context}

Question:
{input}

Answer:
"""

prompt_template = PromptTemplate(input_variables=["context", "input"], template=prompt)
prompt_template

PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template="\nUse the following pieces of context to answer the question at the end.\nDon't try to make up an answer and only use the information you know.\nUse three sentences maximum and keep the answer as concise as possible.\nYou must answer in english.\nContext:\n{context}\n\nQuestion:\n{input}\n\nAnswer:\n")

***Questions***
- The prompt above has 4 regions into which we can insert information. What are these regions? Explain the role of each one.

Tips and tricks: You can modifiy the prompte above to get an answer closer to the one you are looking for.
 

***Answers***
- System prompt: This is the prompt that will be sent to the model. It contains the context and the question.
- Context: This is the information that the model will use to generate its answer. It can be a document, a paragraph, or any other relevant information.
- Question: This is the question that the model is expected to respond to.
- Answer: This is where the model will insert its answer. This is the answer showed to the user.

### Creating the chain

In [ ]:
# Top k of chunks to retrieve from the vectorstore
NB_RETRIVED_CHUNKS = 8

question_answer_chain = create_stuff_documents_chain(llm=llm, prompt=prompt_template)
retriever = vectorstore.as_retriever(
    search_type="mmr", #  Can be “similarity” (default), “mmr”, or “similarity_score_threshold”.
    search_kwargs={
        "k": NB_RETRIVED_CHUNKS,
    }
)
chain_with_retriever = create_retrieval_chain(retriever, question_answer_chain)

In the code above, multiple things go on at the same time.
[create_stuff_documents_chain](https://python.langchain.com/docs/versions/migrating_chains/stuff_docs_chain/) combines documents by concatenating them into a single context window. 

**Questions**:
- What is a retriever? 
- What is the meaning of the parameter: "mmr"? What is the difference between "mmr", "similarity" and "similarity_score_threshold"?
- What is the meaning of the parameter "k" and how increasing/decreasing it may impact the results?
- When using "mmr", there are other parameters for the retriever. One of them is `lambda_mult`. What is the meaning of this parameter and what are the possible values? 
- What `create_retrieval_chain` does?

**Answers**:
- A retriever is a component that retrieves relevant documents from a database based on a query.
- `"mmr"`, `"similarity"` (default), or `"similarity_score_threshold"` define the three search strategies implemented in FAISS.
    - `mmr` stands for Maximal Marginal Relevance. It is a method used to diversify the results of a search query by selecting documents that are both relevant and dissimilar to each other. The "lambda_mult" parameter specifies the degree of diversity among the results (see below).
    - The `similarity` search strategy retrieves documents based on their similarity to the query.
    - The `similarity_score_threshold` also search strategy retrieves documents based on their similarity to the query. However, it allows defining a `score_threshold` parameter, which specifies the minimum similarity score required for a document to be retrieved.
- The parameter `k` specifies the number of documents to retrieve. Increasing `k` will result in more documents being retrieved, while decreasing `k` will result in fewer documents being retrieved. The value of `k` should be chosen based on the desired trade-off between accuracy and performance.
- `lambda_mult` (float) – Number between 0 and 1 that determines the degree of diversity among the results with 0 corresponding to maximum diversity and 1 to minimum diversity. Defaults to 0.5.
- `create_retrieval_chain` creates a chain that retrieves relevant documents from a database based on a query. The chain:
    - uses a retriever to retrieve the relevant documents
    - combines the retrieved documents into a single context window
    - passes this information (as part of the context) to the model to generate the answer to the user's question.

### Chatting with a pdf

Everything is ready to chat with the model: the content of the pdf is embedded in the vectorstore, the model is running, and the chain is created. Let's ask some question to our pdf! Maybe they can help, where the model alone could not.

In [21]:
result = chain_with_retriever.invoke(input={"input": "When is held the conference called AI-days 2025?"})

console.print(Markdown(result["answer"]))

The conference is called AI-DAYS@HES-SO 2025, which means it takes place in January 27-29, 2025.

**Question**:
- What is the size of `result`? What do you expect to find in `result`?
- Let us consider again the "temperature" parameter. For an application that needs to provide accurate information coming from existing document, is it better to use a high or low temperature?
- Suggestion: try to change it and see the difference in the answer.

**Answer**:
- The size of `result` is equal to the parameter `k` discussed above. `result` contains the chunks retrieved by the retriever from the vectorstore. The chunks are ordered by relevance to the query and each chunk has its related metadata. The LLM combines these chunks to generate the final answer (also contained in `result`). To have a better understanding of the `result` variable, just print it!
- The temperature parameter in a language model (LLM) controls the randomness of the model's output. A RAG application, usually, needs to provide accurate information coming from existing documents. In this case, it is better to use a lower temperature value (closer to 0) to make the model more deterministic, favoring higher probability words and resulting in more predictable and repetitive text. This way, the model will be more likely to provide accurate information from the documents.

### Explaining the output

The `result` dictionary contains much more than the simple answer. Thanks to the metadata, we can see the title of the document, the page number, and the context where the answer was found!

In [22]:
console.print(result)

{
    'input': 'When is held the conference called AI-days 2025?',
    'context': [
        Document(
            id='cf193fef-7de6-4ff1-a091-54518acf1262',
            metadata={
                'producer': 'macOS Version 15.1.1 (Build 24B91) Quartz PDFContext',
                'creator': 'Word',
                'creationdate': "D:20250116093703Z00'00'",
                'title': 'Workshop_descriptions',
                'moddate': "D:20250116093703Z00'00'",
                'source': 'data\\PDFs\\Session1.pdf',
                'total_pages': 2,
                'page': 1,
                'page_label': '2'
            },
            page_content='AI-DAYS@HES-SO 2025 –GENEVA & LAUSANNE –JANUARY 27-JANUARY 29, 2025\nWORKSHOP DAY JANUARY
27'
        ),
        Document(
            id='bb2b71cd-ea56-4fc9-bfc0-df419376ae20',
            metadata={
                'producer': 'macOS Version 15.0.1 (Build 24A348) Quartz PDFContext, AppendMode 1.1',
                'creator': 'Word',
                'creationdate': "D:20241105080725Z00'00'",
                'moddate': "D:20241105080730Z00'00'",
                'title': 'Workshop_descriptions',
                'source': 'data\\PDFs\\Session2a.pdf',
                'total_pages': 2,
                'page': 0,
                'page_label': '1'
            },
            page_content='Session 2a: Edge AI Tools, devices, and methods With the advent of Chatbots, LLMs and 
other generative IA technologies, as well as other progresses in the IA ﬁeld, there is an explosion of the demand 
for compute force. IA is no longer computer science; it is computational science. As such, it can no longer be done
with casual, self-managed equipment. More advanced compute infrastructures are required both to satisfy user needs 
(in terms of compute power, GPU Ram capacity) and to ensure a decent'
        ),
        Document(
            id='796d26e1-81d6-4cb0-9837-f5c52ff74116',
            metadata={
                'producer': 'macOS Version 15.1.1 (Build 24B91) Quartz PDFContext',
                'creator': 'Word',
                'creationdate': "D:20250116093703Z00'00'",
                'title': 'Workshop_descriptions',
                'moddate': "D:20250116093703Z00'00'",
                'source': 'data\\PDFs\\Session1.pdf',
                'total_pages': 2,
                'page': 1,
                'page_label': '2'
            },
            page_content="Schedule: 8h30-12h20 + 13h15-17h15 8:30:00 AM 0:05 Opening remarks Sébatien Rumley 
HEIA-FR, HES-SO / Swiss Ai center 8:35:00 AM 0:30 The Alps research infrastructure at CSCS: enabling world-class ML
research in Switzerland Fawzi Mohamed The Swiss National Supercomputing Centre (CSCS), ETH Zurich 9:05:00 AM 0:18 
SCITAS: On-premise and Cloud Infrastructure driving HPC & AI Scientific Computing at EPFL Gilles Fourestey 
SCITAS/EPFL 9:23:00 AM 0:18 Picterra's Infrastructure: Scaling ML for"
        ),
        Document(
            id='24965063-28b5-4108-83a3-32270e0d3140',
            metadata={
                'producer': 'macOS Version 15.2 (Build 24C101) Quartz PDFContext',
                'creator': 'Word',
                'creationdate': "D:20250118184043Z00'00'",
                'title': 'session2b',
                'moddate': "D:20250118184043Z00'00'",
                'source': 'data\\PDFs\\Session2b.pdf',
                'total_pages': 1,
                'page': 0,
                'page_label': '1'
            },
            page_content="Session 2b:  AI for Local Energy Systems  Ce workshop est organisé par le projet phare de
la HES-SO : Smart Energy District. Il vise à présenter quelques projets dans le domaine de l’AI appliquée au 
secteur de l’énergie, en particulier les « local energy communities ». Il s’adresse aux Gestionnaires de Réseaux de
Distribution (GRDs), aux communes et municipalités ainsi qu’aux entreprises qui opèrent dans ce domaine.  This 
workshop is organised by the HES-SO's ﬂagship project: Smart Energy"
        ),
     

---

## 4. Challenge: getting information from a website

Now, it's your turn. Use the code above as inspiration to:
- interact with a website (instead of a PDF)
- create a vectorstore with the information from the website
- chat with the model
- evaluate the output


We suggest to use https://docs.realgames.co/homeio/en/, but you can choose any website you want. We *expect* that your code works with any "normal" website.


Hint: If you need some help to get started check [this](https://python.langchain.com/docs/how_to/document_loader_web/).

### Load the website

Loadgin information from a website can be done in multiple ways. In this solution, we show how to use a `WebBaseLoader` (to load a list of pages) and a `RecursiveUrlLoader` to automatically go deeper from a root url.

As for the pdf files, "loading" means extracting text from the website, deviding it in chunks, and appending it to a list of langchain's [documents](https://python.langchain.com/api_reference/core/documents/langchain_core.documents.base.Document.html).

In [28]:
# All data will be in a new list documents
documents = []

In [29]:
from langchain.document_loaders import WebBaseLoader # read one or a list of pages
from langchain.document_loaders.recursive_url_loader import RecursiveUrlLoader # read recursively from a root page
from langchain.vectorstores.utils import filter_complex_metadata
from bs4 import BeautifulSoup as Soup

is_recurisvely = True

if is_recurisvely:
    url = "https://docs.realgames.co/homeio/en/"
     # you may want to increase the max_depth if you want to get more pages, but then it will take longer to load and (especially) doing the embedding.
    loader = RecursiveUrlLoader(url=url, max_depth=2, extractor=lambda x: Soup(x, "html.parser").text)
    web_docs_up = loader.load()
else:
    urls = [
        "https://spl.hevs.io/spl-docs/index.html",
        "https://spl.hevs.io/syrto/AT-Com/atcom-docs/overview.html",
        "https://spl.hevs.io/syrto/AT-Com/atcom-docs/site/index.html"
        ]
    loader = WebBaseLoader(urls)

    web_docs_up = loader.load()

# tokenize web online  
web_docs = text_splitter.split_documents(web_docs_up)

# we filter out metadata that the embedding below will not be able to manage
documents.extend(filter_complex_metadata(web_docs))
           
print(len(documents))

290


As explained in [Section 3](#what-is-a-document?), a document is composed of two elements: the text and the metadata. The metadata is a dictionary that can contain any information you want to store about the document. In this case, we store the URL of the website.

In [30]:
for doc in documents[:5]:
    print(doc.page_content)

Home I/O















          Skip to content
        















            Home I/O
          



            
              Welcome
            
          

























            Initializing search
          



















    Home I/O
  




          Welcome
          


        Welcome
      



      Table of contents
    



    Home I/O can be used with:
  






        Installation
      




          Manual
Installation
      




          Manual
          




          Manual
        



        Controls
      



        Main Menu
      



        Head-Up Display
      




          Devices
          




          Devices
        



        Devices Map
      



        Device Modes
      



        Lighting
      



        Roller Shades and Up/Down Switch
      



        Garage Door and Entrance Gate
      



        Programmable Remote
Garage Door and Entrance Gate
      



        Programmable Remote
      



        Heater a

In [31]:
for doc in documents[:5]:
    print(doc.metadata)

{'source': 'https://docs.realgames.co/homeio/en/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home I/O', 'description': 'Home I/O Documentation', 'language': 'en'}
{'source': 'https://docs.realgames.co/homeio/en/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home I/O', 'description': 'Home I/O Documentation', 'language': 'en'}
{'source': 'https://docs.realgames.co/homeio/en/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home I/O', 'description': 'Home I/O Documentation', 'language': 'en'}
{'source': 'https://docs.realgames.co/homeio/en/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home I/O', 'description': 'Home I/O Documentation', 'language': 'en'}
{'source': 'https://docs.realgames.co/homeio/en/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home I/O', 'description': 'Home I/O Documentation', 'language': 'en'}


### Adding the website to the vectorstore

We will use the same code as before to add the website to the vectorstore. Here, we store this information in another vectorstore, so we can keep the information from the pdf and the website separated. In some case, we may want to merge the information from different sources. You can check an example [here](https://python.langchain.com/api_reference/community/vectorstores/langchain_community.vectorstores.faiss.FAISS.html).

Here, basically, we used the exact same code as before, but with a different vectorstore.

In [32]:
# we create the vectorstore (it may take a while...)
vectorstore = FAISS.from_documents(documents=documents, embedding=embedding_model)

In [33]:
# we save the vectorstore
VECTORSTORES_DIR = Path("data/vectorstores_web")
vectorstore.save_local(VECTORSTORES_DIR)

### Prompting time!

We reuse the same prompt and retriever and chain as before. Therefore, we can reuse the same chain to answer the question.
However, for a sanity check, we try to ask the same question to the model with and without the website information. Let's see if the RAG is actually useful.


In [34]:
# Simple chain (no context coming from the retrieved documents)
result = simple_llm_chain.invoke(input="List the available detectors in Home I/O.")

console.print(Markdown(result)) # we print the output proposed by the LLM

To list the available detectors in Home I/O, here's a detailed breakdown of the common types:                      

  1 Motion Detectors: These devices detect movement and can be set to arm/disarm based on motion. They're commonly 
    used for security purposes.                                                                                    
  2 Temperature Sensors: These sensors measure temperature levels within a specific range (e.g., 0-100°C or        
    32-212°F). They're useful for monitoring heating/cooling systems, air quality, and more.                       
  3 Humidity Sensors: These devices detect the moisture level in the air, which can be crucial for maintaining     
    indoor humidity levels suitable for various applications like plants, electronics, or even as a backup power   
    source.                                                                                                        
  4 Light Detectors (e.g., LDRs/LCDs): These sensors measure light intensity and are often used to control lighting
    systems based on ambient conditions.                                                                           
  5 Infrared Detectors: Similar to motion detectors but use infrared radiation to detect movement, these can be    
    useful for applications like home automation or security systems that require more precise detection of people 
    or objects.                                                                                                    
  6 Pressure Sensors: These sensors measure changes in pressure within a specific range (e.g., 0-100 kPa). They're 
    used in various applications such as door locks, air quality monitoring, and even as part of smart home        
    automation systems.                                                                                            
  7 Vibration Detectors: These devices detect vibrations or movements that exceed certain thresholds, which can be 
    indicative of potential issues like loose objects, tampering, or other disturbances.                           
  8 Sound Detectors (e.g., Microphones): While not typically classified as a "detector" in the classical sense,    
    sound detectors are used to measure and analyze audio signals from various sources such as voice commands,     
    doorbells, or even environmental sounds like wind or water.                                                    
  9 Air Quality Sensors: These devices monitor parameters like CO2, O3 (ozone), NOx (nitrogen oxides), and other   
    pollutants in the air, providing insights into indoor air quality.                                             
 10 Gas Detectors: These sensors detect specific gases such as carbon monoxide, hydrogen sulfide, or others that   
    can be hazardous to human health or pose a risk to equipment.                                                  
 11 Smoke Detectors: Designed to alert users of potential fires by detecting smoke particles in the air, these     
    devices are an essential component of home safety systems.                                                     
 12 Carbon Monoxide (CO) Detectors: These sensors specifically detect CO levels in the air, which can be hazardous 
    if not addressed promptly.                                                                                     
 13 Ionization Detectors: Also known as ion chambers, these devices measure changes in ionizing radiation from     
    sources like X-rays or gamma rays to detect potential threats.                                                 
 14 Optical Smoke Detectors (OSDs): These sensors use optical principles to detect smoke particles and alert users 
    of a potential fire.                                                                                           
 15 Thermal Imaging Cameras: While not traditional detectors per se, thermal imaging cameras can be used as part of
    an integrated system that detects anomalies in temp

In [41]:
# Chain with the retriever. We need to recreate the chain because the vectorstore has changed.
# The prompt template remains the same.


# Top k of chunks to retrieve from the vectorstore
NB_RETRIVED_CHUNKS = 8

question_answer_chain = create_stuff_documents_chain(llm=llm, prompt=prompt_template)
retriever = vectorstore.as_retriever(
    search_type="mmr", #  Can be “similarity” (default), “mmr”, or “similarity_score_threshold”.
    search_kwargs={
        "k": NB_RETRIVED_CHUNKS,
    }
)
chain_with_retriever = create_retrieval_chain(retriever, question_answer_chain)

In [44]:
result = chain_with_retriever.invoke(input={"input": "List the available detectors in Home I/O."})

console.print(Markdown(result["answer"])) # we print the output proposed by the LLM

Here is a concise answer based on the provided context:                                                            

The available detectors in Home I/O are:                                                                           

 1 Central Alarm and Siren                                                                                         
 2 Smoke Detector                                                                                                  
 3 Brightness Detector                                                                                             
 4 Door Detector                                                                                                   
 5 Thermostat                                                                                                      
 6 Central Alarm                                                                                                   
 7 Lights (roller shades, garage door, entrance gate)                                                              
 8 Heaters

In [43]:
# to get an idea of the context used to answer the question we print the full results with the context.
console.print(result)

{
    'input': 'Which version of Scratch can you use in Home I/O?',
    'context': [
        Document(
            id='a712aeba-0c1e-4494-8779-551b6e4c55b3',
            metadata={
                'source': 'https://docs.realgames.co/homeio/en/scratch2/',
                'content_type': 'text/html; charset=utf-8',
                'title': 'Using with Scratch 2 - Home I/O',
                'description': 'Home I/O Documentation',
                'language': 'en'
            },
            page_content='Home I/O can be used together with Scratch 2 through scratch extensions. On top of this 
page, you can download a Scratch 2 template file which includes the necessary Home I/O extension blocks. After 
opening this file in Scratch, click on More Blocks to see all Home I/O blocks. In order to use Home I/O devices in 
Scratch, they must be set to external mode first.\nA successful connection between Scratch and Home I/O is 
indicated by a green light next to the Home I/O (en) extension.'
        ),
        Document(
            id='e9585540-2dab-4725-b0a6-12318f503461',
            metadata={
                'source': 'https://docs.realgames.co/homeio/en/release-notes/',
                'content_type': 'text/html; charset=utf-8',
                'title': 'Release Notes - Home I/O',
                'description': 'Home I/O Documentation',
                'language': 'en'
            },
            page_content='v1.7.1 (2022/04/05)¶\n\nFixed HTTP command for Up/Down switch on zone L\nAdded tutorial 
on how to use Home I/O and Scratch 3 on different machines\n\nv1.7.0 (2021/10/27)¶\n\nAdded support for Python 
3.9.x\nFixed support for Python 3.8.x.\nChanged licensing model.\n\nv1.6.0 (2020/10/07)¶\n\nAdded support for 
Scratch 3.\nAdded support for Python 3.8.x.\nImproved garage documentation.\nImproved Scratch 2 
documentation.\n\nv1.5.2 (2020/03/19)¶'
        ),
        Document(
            id='a7c0ffff-184d-4b85-bf01-77272f1399ed',
            metadata={
                'source': 'https://docs.realgames.co/homeio/en/scratch3/',
                'content_type': 'text/html; charset=utf-8',
                'title': 'Using with Scratch 3 - Home I/O',
                'description': 'Home I/O Documentation',
                'language': 'en'
            },
            page_content='Thermal Behavior\n      \n\n\n\n\n          SDK\n          \n\n\n\n\n          SDK\n     
\n\n\n\n        Getting Started\n      \n\n\n\n        Memory Addresses\n      \n\n\n\n\n\n\n        System 
Requirements\n      \n\n\n\n        Release Notes\n      \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n      Table of contents\n 
\n\n\n\n    Running Home I/O and Scratch 3 on different machines\n  \n\n\n\n    Home I/O and Scratch 3 Desktop 
(Offline)'
        ),
        Document(
            id='fc4334d5-94e7-4aaa-b395-6cc1c1993dc4',
            metadata={
                'source': 'https://docs.realgames.co/homeio/en/scratch3/',
                'content_type': 'text/html; charset=utf-8',
                'title': 'Using with Scratch 3 - Home I/O',
                'description': 'Home I/O Documentation',
                'language': 'en'
            },
            page_content='Home I/O and Scratch 3 Desktop (Offline)¶\nA version of Scratch 3 Desktop for Home I/O 
(includes all the required extensions) is available for download. This version - Window 10 only - allows to use 
Scratch 3 together with Home I/O without an Internet connection.\nDownload Scratch Desktop Setup 3.12.0'
        ),
        Document(
            id='e85288ab-691f-42e0-beaf-d60efa376ffa',
            metadata={
                'source': 'https://docs.realgames.co/homeio/en/scratch3/',
                'content_type': 'text/html; charset=utf-8',
                'title': 'Using with Scratch 3 - Home I/O',
                'description': 'Home I/O Documentation',
                'language': 'en'
            },
            page_content="Running Home I/O and Scratch 3 on different machines¶\nIt's not mandatory to run

## Conclusion

In this lab, we learned how to create a simple RAG system to answer questions about pdf documents and websites. We used a vector database to store information about the pdf documents and a self-hosted LLM to generate the answers. We also learned how to create prompts and interact with the model to generate text.

Pdf and web documents are just examples of the many applications of RAG systems. RAG can work with any type of document or data source, such as databases, APIs, YouTube videos, excel files, etc. 